# EXP-005: 원본 데이터 활용 + 모델 튜닝 + 안정적 블렌딩

**현재 LB 최고:** XGB 단독 0.96597 (EXP-003C)

| Step | 내용 |
|------|------|
| 1 | 원본 데이터 결합 효과 검증 (XGB 단독) |
| 2 | LightGBM 튜닝 (lr=0.02, 5000est) |
| 3 | CatBoost 재조정 (sample_weight 통일) |
| 4 | 균등 블렌딩 (1/3씩) |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import lightgbm as lgb
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings('ignore')

mpl.rcParams['font.family'] = 'Malgun Gothic'
mpl.rcParams['axes.unicode_minus'] = False

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
orig = pd.read_csv('../data/irrigation_prediction.csv')

TARGET = 'Irrigation_Need'
CAT_COLS = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
            'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

print(f"Train: {train.shape}, Test: {test.shape}, Original: {orig.shape}")

Train: (630000, 21), Test: (270000, 20), Original: (10000, 20)


In [2]:
# 원본 데이터에 id 컬럼 추가 (train과 형식 통일)
orig['id'] = range(len(train), len(train) + len(orig))
orig['is_original'] = 1
train['is_original'] = 0

# 결합
train_combined = pd.concat([train, orig], ignore_index=True)
print(f"결합 후: {train_combined.shape} (train {len(train)} + orig {len(orig)})")
print(f"원본 타겟 분포:\n{orig[TARGET].value_counts()}")

# 타겟 인코딩
target_le = LabelEncoder()
y_combined = target_le.fit_transform(train_combined[TARGET])
y_train_only = target_le.transform(train[TARGET])

# 범주형 인코딩 (train + orig + test 모두 포함)
for col in CAT_COLS:
    le = LabelEncoder()
    all_vals = pd.concat([train_combined[col], test[col]])
    le.fit(all_vals)
    train_combined[col] = le.transform(train_combined[col])
    test[col] = le.transform(test[col])

# sample_weight: 원본 데이터는 0.4 가중치
ORIG_WEIGHT = 0.4
sw_balanced = compute_sample_weight('balanced', y_combined)
is_orig = train_combined['is_original'].values
sw_combined = sw_balanced.copy()
sw_combined[is_orig == 1] *= ORIG_WEIGHT

# OOF 평가는 train 부분만 (원본 제외)
train_mask = train_combined['is_original'] == 0

drop_cols = ['id', TARGET, 'is_original']
feature_cols = [c for c in train_combined.columns if c not in drop_cols]
X_combined = train_combined[feature_cols]
X_test = test[feature_cols]

N_FOLDS = 10
SEED = 42

print(f"\n피처: {len(feature_cols)}개")
print(f"원본 가중치: {ORIG_WEIGHT}")

결합 후: (640000, 22) (train 630000 + orig 10000)
원본 타겟 분포:
Irrigation_Need
Low       5864
Medium    3800
High       336
Name: count, dtype: int64

피처: 19개
원본 가중치: 0.4


## Step 1: 원본 데이터 결합 + XGBoost

In [3]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

xgb_params = {
    'n_estimators': 5000,
    'max_depth': 6,
    'learning_rate': 0.02,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': 0,
    'early_stopping_rounds': 200,
}

# train 부분만으로 CV split, 원본 데이터는 매 fold train에 추가
X_train = X_combined[train_mask].reset_index(drop=True)
y_train = y_combined[train_mask.values]
sw_train = sw_balanced[train_mask.values]  # train의 balanced weight

X_orig = X_combined[~train_mask].reset_index(drop=True)
y_orig = y_combined[~train_mask.values]
sw_orig = sw_combined[~train_mask.values]  # 원본의 낮은 weight

oof_xgb = np.zeros((len(X_train), 3))
test_xgb = np.zeros((len(X_test), 3))

print("=== Step 1: XGBoost + 원본 데이터 ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    # train fold + 원본 데이터 결합
    X_tr = pd.concat([X_train.iloc[tr_idx], X_orig], ignore_index=True)
    y_tr = np.concatenate([y_train[tr_idx], y_orig])
    sw_tr = np.concatenate([sw_train[tr_idx], sw_orig])

    X_val = X_train.iloc[va_idx]
    y_val = y_train[va_idx]

    model = XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr, sample_weight=sw_tr,
              eval_set=[(X_val, y_val)], verbose=0)

    oof_xgb[va_idx] = model.predict_proba(X_val)
    test_xgb += model.predict_proba(X_test) / N_FOLDS
    score = balanced_accuracy_score(y_val, oof_xgb[va_idx].argmax(axis=1))
    print(f"  Fold {fold}: {score:.5f} (best_iter: {model.best_iteration})")

xgb_score = balanced_accuracy_score(y_train, oof_xgb.argmax(axis=1))
print(f"  XGBoost OOF: {xgb_score:.5f} (이전: 0.96812)")

=== Step 1: XGBoost + 원본 데이터 ===
  Fold 0: 0.96742 (best_iter: 4991)
  Fold 1: 0.96717 (best_iter: 4999)
  Fold 2: 0.96786 (best_iter: 4968)
  Fold 3: 0.96999 (best_iter: 4976)
  Fold 4: 0.96868 (best_iter: 4997)
  Fold 5: 0.96956 (best_iter: 4977)
  Fold 6: 0.96937 (best_iter: 4899)
  Fold 7: 0.96694 (best_iter: 4986)
  Fold 8: 0.96844 (best_iter: 4996)
  Fold 9: 0.96694 (best_iter: 4766)
  XGBoost OOF: 0.96824 (이전: 0.96812)


## Step 2: LightGBM 튜닝 + 원본 데이터

In [4]:
oof_lgb = np.zeros((len(X_train), 3))
test_lgb = np.zeros((len(X_test), 3))

print("=== Step 2: LightGBM 튜닝 + 원본 데이터 ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr = pd.concat([X_train.iloc[tr_idx], X_orig], ignore_index=True)
    y_tr = np.concatenate([y_train[tr_idx], y_orig])
    sw_tr = np.concatenate([sw_train[tr_idx], sw_orig])

    X_val = X_train.iloc[va_idx]
    y_val = y_train[va_idx]

    model = LGBMClassifier(
        n_estimators=5000,
        max_depth=-1,
        num_leaves=127,
        learning_rate=0.02,       # 0.04 → 0.02
        subsample=0.8,
        colsample_bytree=0.6,
        reg_alpha=0.1,
        reg_lambda=1.0,
        min_child_samples=20,
        objective='multiclass',
        num_class=3,
        metric='multi_logloss',
        device='gpu',
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    )
    model.fit(X_tr, y_tr, sample_weight=sw_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(200, verbose=False)])

    oof_lgb[va_idx] = model.predict_proba(X_val)
    test_lgb += model.predict_proba(X_test) / N_FOLDS
    score = balanced_accuracy_score(y_val, oof_lgb[va_idx].argmax(axis=1))
    print(f"  Fold {fold}: {score:.5f} (best_iter: {model.best_iteration_})")

lgb_score = balanced_accuracy_score(y_train, oof_lgb.argmax(axis=1))
print(f"  LightGBM OOF: {lgb_score:.5f} (이전: 0.96572)")

=== Step 2: LightGBM 튜닝 + 원본 데이터 ===
  Fold 0: 0.96465 (best_iter: 1778)
  Fold 1: 0.96512 (best_iter: 1802)
  Fold 2: 0.96449 (best_iter: 1723)
  Fold 3: 0.96772 (best_iter: 1916)
  Fold 4: 0.96693 (best_iter: 1836)
  Fold 5: 0.96688 (best_iter: 1802)
  Fold 6: 0.96610 (best_iter: 1860)
  Fold 7: 0.96360 (best_iter: 1716)
  Fold 8: 0.96641 (best_iter: 1943)
  Fold 9: 0.96501 (best_iter: 1797)
  LightGBM OOF: 0.96569 (이전: 0.96572)


## Step 3: CatBoost 재조정 + 원본 데이터

In [5]:
cat_features_idx = [feature_cols.index(c) for c in CAT_COLS]

oof_cat = np.zeros((len(X_train), 3))
test_cat = np.zeros((len(X_test), 3))

print("=== Step 3: CatBoost 재조정 + 원본 데이터 ===")
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr = pd.concat([X_train.iloc[tr_idx], X_orig], ignore_index=True)
    y_tr = np.concatenate([y_train[tr_idx], y_orig])
    sw_tr = np.concatenate([sw_train[tr_idx], sw_orig])

    X_val = X_train.iloc[va_idx]
    y_val = y_train[va_idx]

    model = CatBoostClassifier(
        iterations=3000,
        depth=8,
        learning_rate=0.03,         # 0.04 → 0.03
        l2_leaf_reg=4.0,
        loss_function='MultiClass',
        eval_metric='MultiClass',
        cat_features=cat_features_idx,
        task_type='GPU',
        random_seed=SEED,
        verbose=0,
        early_stopping_rounds=200,
    )
    # sample_weight로 통일 (auto_class_weights 제거)
    model.fit(X_tr, y_tr, sample_weight=sw_tr,
              eval_set=(X_val, y_val))

    oof_cat[va_idx] = model.predict_proba(X_val)
    test_cat += model.predict_proba(X_test) / N_FOLDS
    score = balanced_accuracy_score(y_val, oof_cat[va_idx].argmax(axis=1))
    print(f"  Fold {fold}: {score:.5f} (best_iter: {model.get_best_iteration()})")

cat_score = balanced_accuracy_score(y_train, oof_cat.argmax(axis=1))
print(f"  CatBoost OOF: {cat_score:.5f} (이전: 0.96820)")

=== Step 3: CatBoost 재조정 + 원본 데이터 ===
  Fold 0: 0.96575 (best_iter: 2999)
  Fold 1: 0.96649 (best_iter: 2999)
  Fold 2: 0.96786 (best_iter: 2999)
  Fold 3: 0.97053 (best_iter: 2999)
  Fold 4: 0.96942 (best_iter: 2999)
  Fold 5: 0.96888 (best_iter: 2999)
  Fold 6: 0.96763 (best_iter: 2999)
  Fold 7: 0.96614 (best_iter: 2999)
  Fold 8: 0.96726 (best_iter: 2999)
  Fold 9: 0.96773 (best_iter: 2999)
  CatBoost OOF: 0.96777 (이전: 0.96820)


## Step 4: 결과 비교 + 균등 블렌딩 + 제출

In [6]:
# 개별 모델 비교
print("=" * 55)
print(f"{'모델':<20} {'이전 OOF':>10} {'현재 OOF':>10} {'변화':>10}")
print("=" * 55)
print(f"{'XGBoost':<20} {'0.96812':>10} {xgb_score:>10.5f} {xgb_score-0.96812:>+10.5f}")
print(f"{'LightGBM':<20} {'0.96572':>10} {lgb_score:>10.5f} {lgb_score-0.96572:>+10.5f}")
print(f"{'CatBoost':<20} {'0.96820':>10} {cat_score:>10.5f} {cat_score-0.96820:>+10.5f}")
print("=" * 55)

# 균등 블렌딩 (1/3씩)
blend_equal = (oof_xgb + oof_lgb + oof_cat) / 3
blend_equal_score = balanced_accuracy_score(y_train, blend_equal.argmax(axis=1))

# 참고: 최적 가중치 블렌딩
best_w_score = 0
best_w = None
for w1 in np.arange(0.1, 0.8, 0.05):
    for w2 in np.arange(0.1, 0.8, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < 0.05:
            continue
        blend = w1 * oof_xgb + w2 * oof_lgb + w3 * oof_cat
        s = balanced_accuracy_score(y_train, blend.argmax(axis=1))
        if s > best_w_score:
            best_w_score = s
            best_w = (w1, w2, w3)

print(f"\n균등 블렌딩 OOF:   {blend_equal_score:.5f}")
print(f"최적 블렌딩 OOF:   {best_w_score:.5f} (XGB={best_w[0]:.2f}, LGB={best_w[1]:.2f}, CAT={best_w[2]:.2f})")
print(f"\n이전 블렌딩 OOF:   0.96893")

모델                       이전 OOF     현재 OOF         변화
XGBoost                 0.96812    0.96824   +0.00012
LightGBM                0.96572    0.96569   -0.00003
CatBoost                0.96820    0.96777   -0.00043

균등 블렌딩 OOF:   0.96790
최적 블렌딩 OOF:   0.96860 (XGB=0.40, LGB=0.10, CAT=0.50)

이전 블렌딩 OOF:   0.96893


In [7]:
# 제출 파일 생성
test_blend_equal = (test_xgb + test_lgb + test_cat) / 3
test_blend_opt = best_w[0] * test_xgb + best_w[1] * test_lgb + best_w[2] * test_cat

submissions = {
    'exp005_xgb': test_xgb,
    'exp005_blend_equal': test_blend_equal,
    'exp005_blend_opt': test_blend_opt,
}

for name, preds in submissions.items():
    labels = target_le.inverse_transform(preds.argmax(axis=1))
    sub = pd.DataFrame({'id': test['id'], 'Irrigation_Need': labels})
    path = f'../submissions/submission_{name}.csv'
    sub.to_csv(path, index=False)
    dist = sub['Irrigation_Need'].value_counts()
    print(f"{path}")
    print(f"  {dict(dist)}\n")

../submissions/submission_exp005_xgb.csv
  {'Low': np.int64(159903), 'Medium': np.int64(101164), 'High': np.int64(8933)}

../submissions/submission_exp005_blend_equal.csv
  {'Low': np.int64(159856), 'Medium': np.int64(101217), 'High': np.int64(8927)}

../submissions/submission_exp005_blend_opt.csv
  {'Low': np.int64(159843), 'Medium': np.int64(101120), 'High': np.int64(9037)}

